## CatBoostClassifier

Данные полученных экспериментов находятся в папке `results`.

In [88]:
# %pip install catboost

import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, auc, balanced_accuracy_score, accuracy_score

In [ ]:
df_2020 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2020_preprocessed.csv')
df_merged = pd.read_csv(r'..\..\preprocessing\data\processed\df_merged.csv')

df_2020.head()

,HeartDisease,BMI,SmokerStatus,AlcoholDrinkers,HadStroke,PhysicalHealthDays,MentalHealthDays,DifficultyWalking,Sex,AgeCategory,RaceEthnicityCategory,HadDiabetes,PhysicalActivities,GeneralHealth,SleepHours,HadAsthma,HadKidneyDisease,HadSkinCancer
0,0,16.60,1,0,0,3.0,30.0,0,Female,55-59,White,Yes,1,Very good,5.0,1,0,1
1,0,20.34,0,0,1,0.0,0.0,0,Female,80 or older,White,No,1,Very good,7.0,0,0,0
2,0,26.58,1,0,0,20.0,30.0,0,Male,65-69,White,Yes,1,Fair,8.0,1,0,0
3,0,24.21,0,0,0,0.0,0.0,0,Female,75-79,White,No,0,Good,6.0,0,0,1
4,0,23.71,0,0,0,28.0,0.0,1,Female,40-44,White,No,1,Very good,8.0,0,0,0


In [90]:
results = pd.DataFrame()

### Обучение на наборе 2020 года

In [91]:
df_2020['AgeCategory'] = df_2020['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2020['GeneralHealth'] = df_2020['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

In [92]:
SEED = 67

X = df_2020.drop('HeartDisease', axis=1)
y = df_2020['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [93]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model.fit(
    X_train,
    y_train,
    cat_features=['Sex', 'RaceEthnicityCategory', 'HadDiabetes'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.124346
0:	learn: 0.7994526	test: 0.7969145	best: 0.7969145 (0)	total: 221ms	remaining: 3m 41s
100:	learn: 0.8309849	test: 0.8242785	best: 0.8242785 (100)	total: 24.9s	remaining: 3m 41s
200:	learn: 0.8368839	test: 0.8237190	best: 0.8243028 (121)	total: 51.1s	remaining: 3m 23s
300:	learn: 0.8410960	test: 0.8228361	best: 0.8243028 (121)	total: 1m 19s	remaining: 3m 3s
400:	learn: 0.8450523	test: 0.8217686	best: 0.8243028 (121)	total: 1m 43s	remaining: 2m 35s
500:	learn: 0.8491015	test: 0.8209914	best: 0.8243028 (121)	total: 2m 9s	remaining: 2m 8s
600:	learn: 0.8525523	test: 0.8197095	best: 0.8243028 (121)	total: 2m 34s	remaining: 1m 42s
700:	learn: 0.8558338	test: 0.8186490	best: 0.8243028 (121)	total: 3m	remaining: 1m 17s
800:	learn: 0.8586619	test: 0.8178323	best: 0.8243028 (121)	total: 3m 25s	remaining: 51.1s
900:	learn: 0.8617331	test: 0.8170755	best: 0.8243028 (121)	total: 3m 51s	remaining: 25.4s
999:	learn: 0.8645092	test: 0.8163338	best: 0.8243028 (121)	total:

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [94]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

In [95]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics2020 = {
    'data': '2020',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [96]:
metrics2020

{'data': '2020',
 'model': 'CatBoostClassifier',
 'roc_auc_score': 0.8433346727927207,
 'f1_score': 0.34188846641318127,
 'pr_auc_score': 0.3510545302568115,
 'balanced_accuracy_score': 0.7664086849439846}

### Обучение на общем наборе 2020+2022

In [97]:
df_merged['AgeCategory'] = df_merged['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_merged['GeneralHealth'] = df_merged['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

df_merged = df_merged.fillna('nan')

In [98]:
SEED = 67

X = df_merged.drop(['HeartDisease'], axis=1)
y = df_merged['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [99]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model.fit(
    X_train,
    y_train,
    cat_features=['Sex', 'RaceEthnicityCategory', 'HadDiabetes'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.151162
0:	learn: 0.7993994	test: 0.7986276	best: 0.7986276 (0)	total: 419ms	remaining: 6m 58s
100:	learn: 0.8269847	test: 0.8230076	best: 0.8230076 (100)	total: 41.6s	remaining: 6m 10s
200:	learn: 0.8302671	test: 0.8230097	best: 0.8231171 (164)	total: 1m 27s	remaining: 5m 49s
300:	learn: 0.8331183	test: 0.8226708	best: 0.8231171 (164)	total: 2m 12s	remaining: 5m 8s
400:	learn: 0.8353875	test: 0.8222384	best: 0.8231171 (164)	total: 2m 58s	remaining: 4m 26s
500:	learn: 0.8375663	test: 0.8217327	best: 0.8231171 (164)	total: 3m 46s	remaining: 3m 45s
600:	learn: 0.8395622	test: 0.8214486	best: 0.8231171 (164)	total: 4m 34s	remaining: 3m 2s
700:	learn: 0.8415793	test: 0.8208506	best: 0.8231171 (164)	total: 5m 19s	remaining: 2m 16s
800:	learn: 0.8436708	test: 0.8204281	best: 0.8231171 (164)	total: 6m 10s	remaining: 1m 32s
900:	learn: 0.8453958	test: 0.8201609	best: 0.8231171 (164)	total: 6m 54s	remaining: 45.6s
999:	learn: 0.8470939	test: 0.8197120	best: 0.8231171 (164)

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [100]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics_merged = {
    'data': 'merged',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [101]:
metrics_merged

{'data': 'merged',
 'model': 'CatBoostClassifier',
 'roc_auc_score': 0.8373789861181673,
 'f1_score': 0.34432699083861873,
 'pr_auc_score': 0.3419250040461593,
 'balanced_accuracy_score': 0.7595204857526182}

### Обучение на наборе 2022 года

In [124]:
df_2022 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2022_preprocessed.csv')

df_2022['AgeCategory'] = df_2022['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2022['GeneralHealth'] = df_2022['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

df_2022['RemovedTeeth'] = df_2022['RemovedTeeth'].map({
    'None of them': 0,
    '1 to 5': 1,
    '6 or more, but not all': 2,
    'All': 3
})

df_2022 = df_2022.fillna('nan')

In [125]:
SEED = 67

X = df_2022.drop(['HeartDisease', 'HadAngina', 'HadHeartAttack'], axis=1)
y = df_2022['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

model2022 = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model2022.fit(
    X_train,
    y_train,
    cat_features=['State',
                  'Sex',
                  'LastCheckupTime',
                  'HadDiabetes',
                  'ECigaretteUsage',
                  'RaceEthnicityCategory',
                  'TetanusLast10Tdap',
                  'CovidPos'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.130201
0:	learn: 0.7930649	test: 0.7980683	best: 0.7980683 (0)	total: 507ms	remaining: 8m 26s
100:	learn: 0.8360429	test: 0.8359553	best: 0.8359553 (100)	total: 42s	remaining: 6m 13s
200:	learn: 0.8423029	test: 0.8363931	best: 0.8364837 (143)	total: 1m 25s	remaining: 5m 37s
300:	learn: 0.8471935	test: 0.8364770	best: 0.8365934 (274)	total: 2m 8s	remaining: 4m 58s
400:	learn: 0.8516622	test: 0.8361603	best: 0.8365934 (274)	total: 2m 52s	remaining: 4m 17s
500:	learn: 0.8562241	test: 0.8357573	best: 0.8365934 (274)	total: 3m 36s	remaining: 3m 36s
600:	learn: 0.8599635	test: 0.8354493	best: 0.8365934 (274)	total: 4m 24s	remaining: 2m 55s
700:	learn: 0.8638728	test: 0.8351564	best: 0.8365934 (274)	total: 5m 7s	remaining: 2m 11s
800:	learn: 0.8674107	test: 0.8344957	best: 0.8365934 (274)	total: 5m 51s	remaining: 1m 27s
900:	learn: 0.8706488	test: 0.8340250	best: 0.8365934 (274)	total: 6m 35s	remaining: 43.4s
999:	learn: 0.8735622	test: 0.8337464	best: 0.8365934 (274)	t

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [126]:
y_proba = model2022.predict_proba(X_test)[:, 1]
y_pred = model2022.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics_2022 = {
    'data': '2022',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [128]:
results = pd.DataFrame([
    metrics2020,
    metrics_2022,
    metrics_merged,
])

In [129]:
results.to_csv(r'../results/catboost.csv')